In [ ]:
!mkdir /kaggle/temp

In [ ]:
!wget -O /kaggle/temp/Miniforge3.sh "https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-$(uname)-$(uname -m).sh"

In [ ]:
!rm -rf /kaggle/temp/conda
!bash /kaggle/temp/Miniforge3.sh -b -p /kaggle/temp/conda

In [ ]:
import subprocess
subprocess.run(['/kaggle/temp/conda/bin/mamba', 'install', '-y', 'python=3.13', 'cuda-version=13.0', 'pytorch=2.11',
                'pillow', 'matplotlib', 'scipy', 'nvimgcodec', 'nccl', 'cuda-cudart-dev'], check=True)
subprocess.run(['/kaggle/temp/conda/bin/python', '-c',
                'import sys, torch, triton; print(sys.version); print(torch.__version__, torch.version.cuda, torch.cuda.nccl.version(), triton.__version__)'],
               check=True)

In [ ]:
!rm -rf /kaggle/temp/music_deep
!git clone -b exp-038-phase-b-joint-refiner https://github.com/kwon-young/music_deep.git /kaggle/temp/music_deep
!git -C /kaggle/temp/music_deep log --oneline -1

In [ ]:
import os
# The COCO JSON is parsed fresh (~1 min) with the current code.
os.makedirs('/kaggle/temp/cache', exist_ok=True)

# Attach a model version holding the Exp 037 detector and its Phase A refiner, with this layout:
#   037_029_recipe_from_023/train_detection/checkpoints/latest_model.pt
#   037_029_recipe_from_023/refiner_phase_a/refiner.pt
print(sorted(os.listdir('/kaggle/input')))
wanted = {
    '/kaggle/temp/init_037.pt': os.path.join('037_029_recipe_from_023', 'train_detection', 'checkpoints', 'latest_model.pt'),
    '/kaggle/temp/refiner_037_phase_a.pt': os.path.join('037_029_recipe_from_023', 'refiner_phase_a', 'refiner.pt'),
}
found = {link: [] for link in wanted}
for root, dirs, files in os.walk('/kaggle/input'):
    dirs[:] = [d for d in dirs if d not in ('datasets', 'ssl-imslp', 'trompa-coco', 'trompa-coco-cache')]
    for name in files:
        path = os.path.join(root, name)
        for link, suffix in wanted.items():
            if path.endswith(suffix):
                found[link].append(path)
print(found)
for link, paths in found.items():
    assert len(paths) == 1, (link, paths)
    if os.path.lexists(link):
        os.remove(link)
    os.symlink(paths[0], link)

In [ ]:
# Exp 038: Exp 037 continued with the line endpoint refiner trained inside the detector step
# (Exp 035 Phase B), on the 46 training pieces (20% of the pieces held out, seed 0, as Phase A).
!PYTHONPATH=/kaggle/temp/music_deep /kaggle/temp/conda/bin/mamba run timeout 11h torchrun --nproc_per_node=2 /kaggle/temp/music_deep/src/train_detection.py \
    --exp_dir experiments/038_phase_b_joint_refiner \
    --patch_size 64 \
    --epochs 30 \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --headless \
    --cache_dir /kaggle/temp/cache/ \
    --use_sdpa \
    --log_epoch_interval 1 \
    --warmup_epochs 1 \
    --lr 2e-5 \
    --compile \
    --extract_patch_min 54 \
    --extract_patch_max 74 \
    --class_weight_variance_lambda 0.5 \
    --class_weight_size_difficulty_max 8.0 \
    --holdout_fraction 0.2 \
    --holdout_seed 0 \
    --detector_checkpoint /kaggle/temp/init_037.pt \
    --line_refiner \
    --refiner_init /kaggle/temp/refiner_037_phase_a.pt

In [ ]:
# All 351 pages with the full line pipeline (refine -> NMS 4 px -> noisy-OR); the refiner is read
# from the same checkpoint. Evaluate locally on all pages and on held_out_image_ids.json.
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint experiments/038_phase_b_joint_refiner/train_detection/checkpoints/latest_model.pt \
    --refiner_checkpoint experiments/038_phase_b_joint_refiner/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/038_phase_b_joint_refiner/inference_full_pipeline \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa

In [ ]:
# Optional (time permitting): coarse head only, the old top-k path, to separate head and refiner changes.
!PYTHONPATH=/kaggle/temp/music_deep/src /kaggle/temp/conda/bin/mamba run python /kaggle/temp/music_deep/src/inference_coco.py \
    --checkpoint experiments/038_phase_b_joint_refiner/train_detection/checkpoints/latest_model.pt \
    --output_dir experiments/038_phase_b_joint_refiner/inference \
    --anno_path ../input/datasets/kwonyoungchoi/trompa-coco/annotations/instances_trainval2017.json \
    --img_dir ../input/datasets/kwonyoungchoi/trompa-coco/trainval2017 \
    --cache_dir /kaggle/temp/cache/ \
    --patch_size 64 \
    --use_sdpa